# Реаугментация Stage 2 + Stage 3

Ноутбук запускает новый пайплайн:
1. Stage 2 v2: light paraphrase через `stage2_paraphrase_v2.py`
2. Stage 3 v2: deep paraphrase через `stage3_paraphrase_v2.py`
3. Проверка SVM / Logistic Regression / Naive Bayes / rubert-tiny2 до и после аугментации
4. Финальная проверка `Data/data_after_stage3.csv`

Файлы остаются обычными: `data_after_stage2.csv` и `data_after_stage3.csv`.

___
## Подготовка окружения

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Локальный запуск: Google Drive не монтируется')

In [ ]:
REPO_URL = 'https://github.com/KVTur23/Mifi_VKR.git'
REPO_BRANCH = 'new-reaugment'

if IN_COLAB:
    REPO_DIR = Path('/content/VKR')
    PROJECT_ROOT = REPO_DIR / 'code'

    DRIVE_ROOT = Path('/content/drive/MyDrive/VKR')
    DRIVE_DATA = DRIVE_ROOT / 'Data'
    DRIVE_RESULTS = DRIVE_ROOT / 'results'
    DRIVE_LOGS = DRIVE_ROOT / 'logs'
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
    REPO_DIR = PROJECT_ROOT.parent

print(f'PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
def run_cmd(cmd, cwd=None):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=True)


def link_to_drive(name, drive_path):
    local_path = PROJECT_ROOT / name
    drive_path.parent.mkdir(parents=True, exist_ok=True)

    drive_missing_or_empty = not drive_path.exists() or not any(drive_path.iterdir())
    if drive_missing_or_empty:
        if local_path.exists() and not local_path.is_symlink():
            if drive_path.exists():
                shutil.rmtree(drive_path)
            shutil.copytree(local_path, drive_path)
        else:
            drive_path.mkdir(parents=True, exist_ok=True)

    if local_path.is_symlink() or local_path.is_file():
        local_path.unlink()
    elif local_path.exists():
        shutil.rmtree(local_path)

    local_path.symlink_to(drive_path, target_is_directory=True)
    print(f'{name}: {local_path} -> {drive_path}')


if IN_COLAB:
    if (REPO_DIR / '.git').exists():
        print('Репозиторий уже есть — обновляю через git pull')
        run_cmd(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
        run_cmd(['git', '-C', str(REPO_DIR), 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'])
        run_cmd(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])
    else:
        print(f'Клонирую {REPO_URL} в {REPO_DIR}')
        run_cmd(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, str(REPO_DIR)])

    link_to_drive('Data', DRIVE_DATA)
    link_to_drive('results', DRIVE_RESULTS)
    link_to_drive('logs', DRIVE_LOGS)

    commit = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print(f'\nТекущий коммит: {commit}')
else:
    print('Локально используется текущая рабочая копия')

In [ ]:
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%cd {PROJECT_ROOT}

import re
import pandas as pd

from src.utils.data_loader import (
    DATA_DIR, TEXT_COL, LABEL_COL, get_class_distribution,
)
from src.utils.pipeline_config import load_pipeline_config

try:
    run_cmd(['nvidia-smi'])
except FileNotFoundError:
    print('nvidia-smi не найден')

GPU = 'A100_40'
pipeline_cfg = load_pipeline_config(GPU)

CONFIG_PATH = str(PROJECT_ROOT / 'config_models' / 'aug_configs' / 'model_vllm_32b.json')
print(f'Конфиг модели: {CONFIG_PATH}')

___
## Функции оценки классификаторов

In [ ]:
import gc
import random
import numpy as np
import torch
from transformers import set_seed
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from src.classification.embeddings import prepare_features
from src.classification.evaluate import evaluate_model
from src.classification.rubert_classifier import train_and_evaluate

EVAL_SEED = 42
RUBERT_TINY2_MODEL = 'cointegrated/rubert-tiny2'
RUBERT_TINY2_EPOCHS = 15
RUBERT_TINY2_BATCH_SIZE = 32
RUBERT_TINY2_LR = 5e-4


def _set_eval_seed(seed=EVAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def _prepare_tfidf_eval(df_train, df_test):
    X_train, y_train_raw, X_test, y_test_raw = prepare_features(
        df_train, df_test, use_cache=False,
    )
    le = LabelEncoder()
    y_train = le.fit_transform(y_train_raw)
    y_test = le.transform(y_test_raw)
    return X_train, y_train, X_test, y_test, le.classes_


def evaluate_classic_models(df_train, df_test, prefix):
    X_train, y_train, X_test, y_test, label_names = _prepare_tfidf_eval(df_train, df_test)
    results = []

    results.append(evaluate_model(
        name=f'[{prefix}] Linear SVM',
        estimator=LinearSVC(max_iter=10000, random_state=EVAL_SEED, dual='auto'),
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        label_names=label_names,
        param_grid={'C': [0.01, 0.1, 1, 10]},
    ))

    results.append(evaluate_model(
        name=f'[{prefix}] Logistic Regression',
        estimator=LogisticRegression(solver='lbfgs', max_iter=1000, random_state=EVAL_SEED),
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        label_names=label_names,
        param_grid={'C': [0.01, 0.1, 1, 10]},
    ))

    results.append(evaluate_model(
        name=f'[{prefix}] Multinomial Naive Bayes',
        estimator=MultinomialNB(),
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        label_names=label_names,
        param_grid={'alpha': [0.01, 0.1, 0.5, 1.0]},
    ))

    return results


def evaluate_rubert_tiny2(df_train, df_test, prefix):
    _set_eval_seed()
    result = train_and_evaluate(
        df_train=df_train,
        df_test=df_test,
        model_name=RUBERT_TINY2_MODEL,
        lr=RUBERT_TINY2_LR,
        num_epochs=RUBERT_TINY2_EPOCHS,
        batch_size=RUBERT_TINY2_BATCH_SIZE,
        name=f'[{prefix}] rubert-tiny2',
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def evaluate_all_models(df_train, df_test, prefix):
    results = evaluate_classic_models(df_train, df_test, prefix)
    results.append(evaluate_rubert_tiny2(df_train, df_test, prefix))
    return results


def results_to_frame(results, stage):
    rows = []
    for r in results:
        model = r['name'].split('] ', 1)[1] if '] ' in r['name'] else r['name']
        rows.append({
            'stage': stage,
            'model': model,
            'balanced_accuracy': round(float(r['balanced_accuracy']), 4),
            'macro_f1': round(float(r['macro_f1']), 4),
        })
    return pd.DataFrame(rows)


def save_model_check_results(*frames):
    results_dir = PROJECT_ROOT / 'results'
    results_dir.mkdir(exist_ok=True)
    df_results = pd.concat(frames, ignore_index=True)
    csv_path = results_dir / 'reaugmentation_model_checks.csv'
    df_results.to_csv(csv_path, index=False)
    print(f'Сохранено: {csv_path}')
    return df_results

___
## Проверка входных данных

In [ ]:
required_files = [
    DATA_DIR / 'train_after_eda.csv',
    DATA_DIR / 'data_after_stage1.csv',
    DATA_DIR / 'data_test.csv',
]
missing = [p for p in required_files if not p.exists()]
assert not missing, 'Не найдены входные файлы: ' + ', '.join(str(p) for p in missing)

for path in required_files:
    df_tmp = pd.read_csv(path)
    print(f'{path.name}: {len(df_tmp)} строк, {df_tmp[LABEL_COL].nunique()} классов')

___
## Baseline на неаугментированных данных

In [ ]:
df_train_baseline = pd.read_csv(DATA_DIR / 'train_after_eda.csv')
df_test_eval = pd.read_csv(DATA_DIR / 'data_test.csv')

baseline_results = evaluate_all_models(
    df_train=df_train_baseline,
    df_test=df_test_eval,
    prefix='Baseline',
)

df_baseline_checks = results_to_frame(baseline_results, stage='baseline')
save_model_check_results(df_baseline_checks)
display(df_baseline_checks)

In [ ]:
RESET_STAGE23 = True

if RESET_STAGE23:
    for name in ['data_after_stage2.csv', 'data_after_stage3.csv']:
        path = DATA_DIR / name
        if path.exists():
            path.unlink()
            print(f'Удалён старый файл: {path}')
        else:
            print(f'Файл уже отсутствует: {path}')

___
## Stage 2 v2: Light Paraphrase

In [ ]:
from src.augmentation.stage2_paraphrase_v2 import main as run_stage2

run_stage2(CONFIG_PATH, pipeline_cfg=pipeline_cfg)

In [ ]:
df_after_s2 = pd.read_csv(DATA_DIR / 'data_after_stage2.csv')
dist_s2 = get_class_distribution(df_after_s2)

print(f'Записей после Stage 2 v2: {len(df_after_s2)}')
print(f'Классов с < 50 примерами: {(dist_s2 < 50).sum()}')
print(f'Min per-class: {dist_s2.min()}')

___
## Stage 3 v2: Deep Paraphrase

In [ ]:
from src.augmentation.stage3_paraphrase_v2 import main as run_stage3

run_stage3(CONFIG_PATH, pipeline_cfg=pipeline_cfg)

___
## Финальная валидация

In [ ]:
df = pd.read_csv(DATA_DIR / 'data_after_stage3.csv')
df_test = pd.read_csv(DATA_DIR / 'data_test.csv')

broken_re = re.compile(
    r'<\d+>?|<[A-Z]{2,5}\b|\bNEFT\b|\bGAZ\b|\bOKPO\b|\bOGRN\b|\bINN\b|\bKPP\b|\bBIK\b'
)
broken_count = df[TEXT_COL].str.contains(broken_re, regex=True, na=False).sum()
assert broken_count == 0, f'Найдено {broken_count} broken-строк'
print('Broken placeholders: 0')

class_counts = df[LABEL_COL].value_counts()
min_count = class_counts.min()
print(f'Min per-class: {min_count} (target >= 40)')
assert min_count >= 40, f'Класс {class_counts.idxmin()} имеет только {min_count} примеров'

aug_median = df[TEXT_COL].str.len().median()
test_median = df_test[TEXT_COL].str.len().median()
length_ratio = abs(aug_median - test_median) / test_median
print(f'Median length: aug={aug_median:.0f}, test={test_median:.0f}, ratio={length_ratio:.2%}')
assert length_ratio < 0.20, 'Расхождение медиан длин >20%'
print('Length distribution close to test')

test_texts = set(df_test[TEXT_COL].dropna().tolist())
overlap = df[df[TEXT_COL].isin(test_texts)]
assert len(overlap) == 0, f'Найдено {len(overlap)} дубликатов с test (data leak)'
print('No test data leak')

print('\n=== ИТОГОВАЯ СТАТИСТИКА ===')
print(f'Total rows: {len(df)}')
print(f'Classes: {df[LABEL_COL].nunique()}')
print(f'Per-class: min={class_counts.min()}, max={class_counts.max()}, mean={class_counts.mean():.0f}')
print(f'Length: median={aug_median:.0f}, p95={df[TEXT_COL].str.len().quantile(0.95):.0f}')

___
## Проверка моделей после аугментации

In [ ]:
augmented_results = evaluate_all_models(
    df_train=df,
    df_test=df_test,
    prefix='Augmented',
)

df_augmented_checks = results_to_frame(augmented_results, stage='augmented')
df_model_checks = save_model_check_results(df_baseline_checks, df_augmented_checks)

display(df_model_checks)

comparison = df_model_checks.pivot(index='model', columns='stage', values=['balanced_accuracy', 'macro_f1'])
comparison[('delta', 'balanced_accuracy')] = (
    comparison[('balanced_accuracy', 'augmented')] - comparison[('balanced_accuracy', 'baseline')]
)
comparison[('delta', 'macro_f1')] = (
    comparison[('macro_f1', 'augmented')] - comparison[('macro_f1', 'baseline')]
)
display(comparison.sort_index())